# 00b · Tokenise the FULL corpus (all ~4.5B tokens)
Notebook `00` must have run first, because this reuses its tokenizer and its train/validation split.

Every document of all four sources is streamed and tokenised into `train_full.bin` (about 9 GB). It holds out
**exactly the same validation documents** as the 10% sample: the same random draws are replayed, and the
held-out docs are re-tokenised and compared byte-for-byte with the existing `val.bin`. So `val.bin` and
`val_prompts.json` stay unseen, and validation losses stay comparable with the 10% runs.



In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
!pip -q install ijson "huggingface_hub[hf_xet]" tokenizers
import os
from huggingface_hub import login, whoami

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    p = f'{CODE_DIR}/hf_token.txt'
    if os.path.exists(p):
        return open(p).read().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as e:
        raise RuntimeError(f'No HF token found: set $HF_TOKEN or create {p}') from e

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in to Hugging Face as', whoami(token=HF_TOKEN)['name'])

In [ ]:
from huggingface_hub import hf_hub_download
REPO = 'Exploration-Lab/CS-781-Capstone'
RAW = '/content/raw'
SOURCES = {'acts': 'train_acts.json', 'high_courts': 'train_high_courts.json',
           'supreme_court': 'train_supreme_court.json', 'tribunals': 'train_tribunals.json'}
files = {}
for name, fn in SOURCES.items():
    files[name] = hf_hub_download(REPO, f'Phase_1_Pretraining/Train/{fn}', repo_type='dataset', local_dir=RAW,
                                  token=HF_TOKEN)
    print(name, files[name])
test_path = hf_hub_download(REPO, 'Phase_1_Pretraining/Test/test.json', repo_type='dataset', local_dir=RAW,
                            token=HF_TOKEN)
!du -sh {RAW}/Phase_1_Pretraining/*/*

In [ ]:
# reuse the tokenizer, val.bin and the exact sampling parameters from notebook 00
import json, shutil, filecmp
from lm_lib import data_prep as dp
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
for f in ['tokenizer.json', 'val.bin', 'data_meta.json']:
    shutil.copyfile(f'{DRIVE_DATA}/{f}', f'{WORK}/{f}')
meta = json.load(open(f'{WORK}/data_meta.json'))
print('sampling params from 00:', {k: meta.get(k) for k in ['frac', 'val_frac', 'normalize_ws']})
print('source order:', list(files))   # must be the same order as in 00 (it is: same SOURCES dict)

In [ ]:
full_stats = dp.tokenize_full_corpus(files, WORK, f'{WORK}/tokenizer.json', sample_frac=meta['frac'],
                                     val_frac=meta['val_frac'], seed=1337,
                                     normalize_whitespace=meta.get('normalize_ws', False))
same = filecmp.cmp(f'{WORK}/val.bin', f'{WORK}/val_check.bin', shallow=False)
print('held-out docs identical to the 10% run val.bin:', same)
assert same, 'validation split does not match notebook 00 -- do NOT use this train_full.bin'
!ls -lh {WORK}/train_full.bin

In [ ]:
!df -h /content/drive | tail -1
shutil.copyfile(f'{WORK}/train_full.bin', f'{DRIVE_DATA}/train_full.bin')
shutil.copyfile(f'{WORK}/full_stats.json', f'{DRIVE_DATA}/full_stats.json')
drive.flush_and_unmount()   # make sure the 9 GB file is fully written to Drive before the runtime goes away
print('done')